## This notebook is for the handling and manipulation of the molecule database

## After Retrieval from the ChEMBL database

### Combine individual target databases in one chemical library 

In [1]:
import pandas as pd
import os

def combine_chembl_tsvs(file_list, output_filename="combined_chembl_database.csv"):
    columns_to_keep = [
        'Molecule ChEMBL ID', # In TSV, ChEMBL often uses "Molecule ChEMBL ID" instead of "Compound ChEMBL ID"
        'Molecular Weight', 
        'Polar Surface Area', 
        'HBA', 
        'HBD', 
        '#Rotatable Bonds', 
        'Aromatic Rings', 
        'Heavy Atoms', 
        'Molecular Formula', 
        'Smiles', 
        'Inchi Key', 
        'Inchi'
    ]
    
    dataframes = []
    
    for file in file_list:
        if not os.path.exists(file):
            print(f"Warning: File '{file}' not found. Skipping.")
            continue
            
        print(f"Processing {file}...")
        
        try:
            # Strictly read as Tab-Separated Values. No guessing. No dropping lines.
            df = pd.read_csv(file, sep='\t', low_memory=False)
            
            # Handle naming variations between ChEMBL export versions
            if 'Molecule ChEMBL ID' not in df.columns and 'Compound ChEMBL ID' in df.columns:
                df.rename(columns={'Compound ChEMBL ID': 'Molecule ChEMBL ID'}, inplace=True)
            if '#Rotatable Bonds' not in df.columns and 'Rotatable Bonds' in df.columns:
                df.rename(columns={'Rotatable Bonds': '#Rotatable Bonds'}, inplace=True)
                
            # Filter the dataframe to keep only the requested columns
            available_cols = [col for col in columns_to_keep if col in df.columns]
            df = df[available_cols]
            
            # Ensure all requested columns exist, filling missing ones with NA
            for col in columns_to_keep:
                if col not in df.columns:
                    df[col] = pd.NA
                    
            # Reorder to match your exact request
            df = df[columns_to_keep]
            dataframes.append(df)
            print(f"  -> Successfully loaded {len(df)} rows.")
            
        except Exception as e:
            print(f"  -> Error reading {file}: {e}")
            
    # Combine all the dataframes into one
    if dataframes:
        combined_df = pd.concat(dataframes, ignore_index=True)
        print(f"\nTotal rows before deduplication: {len(combined_df)}")
        
        # Remove identical molecules
        combined_df.drop_duplicates(subset=['Inchi Key'], keep='first', inplace=True)
        combined_df.drop_duplicates(subset=['Smiles'], keep='first', inplace=True)
        
        print(f"Total unique molecules after deduplication: {len(combined_df)}")
        
        # Save the final output to a standard CSV (since you are now in control of the formatting)
        combined_df.to_csv(output_filename, index=False)
        print(f"Successfully saved to {output_filename}")
    else:
        print("No valid data was extracted. Please check your file names.")

# --- HOW TO RUN ---
# Put your newly downloaded TSV files in the same folder as this script
my_chembl_files = [
    "ChEMBL_Ac.tsv",
    "ChEMBL_Nf.tsv",
    "ChEMBL_Tb.tsv",
    "ChEMBL_Tc.tsv",
    "ChEMBL_Tbr.tsv",
    "ChEMBL_Tbg.tsv"
]

combine_chembl_tsvs(file_list=my_chembl_files, output_filename="combined_chembl_database.csv")

Processing ChEMBL_Ac.tsv...
  -> Successfully loaded 44 rows.
Processing ChEMBL_Nf.tsv...
  -> Successfully loaded 150 rows.
Processing ChEMBL_Tb.tsv...
  -> Successfully loaded 74505 rows.
Processing ChEMBL_Tc.tsv...
  -> Successfully loaded 82347 rows.
Processing ChEMBL_Tbr.tsv...
  -> Successfully loaded 5449 rows.
Processing ChEMBL_Tbg.tsv...
  -> Successfully loaded 361 rows.

Total rows before deduplication: 162856
Total unique molecules after deduplication: 92593
Successfully saved to combined_chembl_database.csv
